# 13차시 실습 — 계획·반성 루프 & 내 MVP에 통합 (Day2 종합)

> **day1 연속 실습.** day1에서 만든 추천 앱(공통 예시 **"맛집 추천 도우미"**)을, 이번엔 **스스로 계획하고 스스로 고쳐 쓰는** 에이전트로 키웁니다.
> Day2의 마지막 실습 — 새 부품을 배우는 게 아니라, **모은 부품(도구·RAG·메모리·멀티)을 내 앱에 끼우는** 날입니다.

1. **Plan → Execute** : 큰 목표를 하위작업으로 **분해**하고 한 단계씩 실행 (같은 MVP에 새 기능 설계)
2. **Reflection 루프** : MVP가 내보낼 답을 만들고 → **스스로 비평**하고 → **개선** (점수로 향상 확인)
3. **내 팀 MVP에 적용** : day1 팀 MVP에 붙일 에이전트 기능 **1개 + 반성 루프** 결정

## 🧪 사용법
- 워크샵 본 실습은 **Codex CLI**로, 이 노트북은 같은 개념을 **OpenAI API로 즉시 실행**해 보는 동반 자료입니다.
- 각 단계에 **⌨️ 터미널 Codex** 명령(복붙용)을 함께 적어 두었습니다.
- 위에서부터 **순서대로** 실행하세요 (`Shift+Enter`).

In [7]:
# ✅ 환경 준비 — 맨 처음 한 번만 (day1 노트북과 동일)
%pip install -q openai
import os, json, getpass
from openai import OpenAI
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY (화면에 안 보임): ")
client = OpenAI(); MODEL = "gpt-4o-mini"
print("준비 완료 ·", MODEL)


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
준비 완료 · gpt-4o-mini


## 오늘의 연속 시나리오 — 같은 MVP, 더 똑똑하게

- **day1**: 버튼 누르면 메뉴를 **랜덤 추천**하는 앱 (정해진 동작).
- **7차시(day2)**: 같은 앱을 **ReAct 에이전트**로 — 도구를 스스로 골라 답하게 만들었습니다.
- **오늘(13차시)**: 같은 **"맛집 추천 도우미" MVP**에 ① 한 번에 못 푸는 일을 **계획**으로 끝까지, ② 답 품질을 **반성**으로 끌어올립니다.

> 구조는 동일하니 ** 팀 MVP**로 바꾸면 그대로 적용됩니다 (마지막 5단계).

## 1단계 — Plan → Execute (목표를 하위작업으로 분해)

에이전트는 **여러 단계를 거치는** 존재입니다. 큰 목표를 받으면 먼저 **하위작업으로 쪼개고**(Planning), 한 단계씩 실행하며 **지금까지 한 일**을 다음 단계에 넘깁니다(= Least-to-Most의 누적).
> 핵심 순서: **계획이 먼저, 실행이 나중.**

여기서는 우리 MVP(맛집 추천 도우미)에 **새 기능을 설계**하는 일을 계획으로 풀어 봅니다.

⌨️ **터미널 Codex:** `codex "맛집 추천 도우미에 '주간 인기 맛집 리포트' 기능을 추가하는 작업을 3~5개 하위작업으로 분해한 뒤 한 단계씩 실행하는 plan-execute 루프를 만들어줘"`

In [8]:
def make_plan(goal: str):
    """목표 -> 3~5개의 순차적 하위작업(JSON)으로 분해 (Planning)."""
    resp = client.chat.completions.create(
        model=MODEL,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": '너는 계획 수립가다. 사용자의 목표를 3~5개의 구체적이고 순차적인 하위작업으로 분해하라. 반드시 {"steps": ["...", "..."]} 형태의 JSON으로만 답하라.'},
            {"role": "user", "content": f"목표: {goal}"},
        ],
    )
    return json.loads(resp.choices[0].message.content)["steps"]

def execute_step(goal: str, step: str, done: str):
    """전체 목표와 지금까지 한 일을 참고해 '현재 하위작업만' 수행 (Execute)."""
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "너는 실행가다. 전체 목표와 지금까지 한 일을 참고해, 이번 하위작업만 간결히 수행하라."},
            {"role": "user", "content": f"전체 목표: {goal}\n\n지금까지 한 일:\n{done or '(없음)'}\n\n이번 하위작업: {step}\n\n결과를 3~4문장으로."},
        ],
    )
    return resp.choices[0].message.content.strip()

# 같은 MVP에 새 기능을 설계하는 목표
goal = "맛집 추천 도우미 MVP에 '예산·취향 기반 3코스(전채→메인→후식) 추천' 기능을 추가한다"

plan = make_plan(goal)
print("📋 계획(하위작업):")
for i, s in enumerate(plan, 1):
    print(f"  {i}. {s}")

print("\n▶ 단계 실행 ↓")
done = ""
for i, step in enumerate(plan, 1):
    out = execute_step(goal, step, done)
    print(f"\n=== 단계 {i}: {step} ===\n{out}")
    done += f"- {step}: {out}\n"   # 앞 단계 결과를 다음 단계에 누적 (Least-to-Most)

📋 계획(하위작업):
  1. 현재 사용 중인 맛집 추천 도우미의 기능 분석
  2. 예산 및 취향 기반의 3코스 추천 로직 설계
  3. 추천 기능을 MVP에 통합하여 테스트 및 피드백 받기
  4. 사용자 인터페이스(UI) 디자인 개선 및 최종 수정
  5. 베타 테스트 후 최종 배포 준비 및 마케팅 전략 수립

▶ 단계 실행 ↓

=== 단계 1: 현재 사용 중인 맛집 추천 도우미의 기능 분석 ===
현재 사용 중인 맛집 추천 도우미는 사용자로부터 입력받은 데이터(위치, 요리 종류, 선호 음식)를 바탕으로 추천 알고리즘을 통해 맛집 정보를 제공합니다. 그러나 예산이나 취향을 반영한 3코스 추천 기능은 포함되어 있지 않아, 사용자 맞춤화가 부족합니다. 이 기능을 추가하면 보다 개인화된 식사 경험을 제공할 수 있을 것으로 보입니다. 따라서 현재 시스템의 추천 방식에 예산 및 취향 기반의 3코스 구성 요소를 통합할 필요가 있습니다.

=== 단계 2: 예산 및 취향 기반의 3코스 추천 로직 설계 ===
예산 및 취향 기반의 3코스 추천 로직은 사용자 입력을 통해 예산 범위와 특정 취향(예: 매운 음식, 비건, 해산물 등)을 반영하여 구성됩니다. 먼저, 전체 데이터베이스에서 예산에 맞는 식당 목록을 필터링한 후, 각 식당의 전채, 메인, 후식을 고려하여 조합 가능한 코스를 평가합니다. 추천할 코스는 각 식당의 평점과 사용자 선호도를 기반으로 하여 점수를 매기고, 최고 점수를 받은 식당들을 3코스로 조합하여 최종 추천 리스트를 작성합니다. 이 과정을 통해 사용자 맞춤형 추천을 제공함으로써 식사 경험을 향상시킵니다.

=== 단계 3: 추천 기능을 MVP에 통합하여 테스트 및 피드백 받기 ===
추천 기능을 MVP에 통합하여 예산 및 취향 기반의 3코스 추천 로직을 구현했습니다. 사용자가 입력한 예산 범위와 선호 음식에 따라 필터링된 식당 목록에서 전채, 메인, 후식을 조합하여 추천 코스를 생성합니다. 이제 해당 기능을 테스트하고 사용자로부터 피드백을 받아 최종 조

### 정리 — 1단계

- **분해(Planning)** 가 먼저, **실행(Execute)** 이 나중. 각 단계는 앞 단계 결과(`done`)를 **누적**해서 받습니다(= Least-to-Most).
- 계획은 고정이 아닙니다. 새 정보가 생기면 **재계획**할 수 있어야 진짜 에이전트입니다.
- `goal`을 **우리 팀 도메인**으로 바꿔 보세요 (예: "instalite에 주간 인기 게시물 요약을 만든다", "가계부에 월말 지출 리포트를 만든다").

## 2단계 — Reflection 루프 (생성 → 비평 → 개선)

한 번에 완벽한 답은 드뭅니다. **초안을 만들고 → 스스로 비평하고 → 고쳐 쓰면** 품질이 오릅니다(Self-Refine).
향상을 **숫자로** 보기 위해 별도의 LLM이 **체크리스트 rubric**(7개 항목)으로 채점합니다 — 항목별 충족 개수가 점수라, 라운드마다 어떤 항목이 채워졌는지 눈에 보입니다.

> ⚙️ 초안은 일부러 **한두 문장**으로 얇게 만듭니다 — 개선 여지를 남겨야 반성 효과가 관찰됩니다.

⌨️ **터미널 Codex:** `codex "맛집 추천 메시지를 rubric 체크리스트로 채점하는 self-refine 루프를 만들어줘 (초안은 얇게, 비평은 rubric 기반)"`


In [9]:
# rubric 체크리스트 — 각 항목이 답에 있으면 1점, 없으면 0점 (총 7점)
RUBRIC = [
    "구체적인 식당 이름 명시",
    "가격 또는 1인당 예상 비용 숫자로 명시",
    "매운 정도(맵기 단계·시그니처 메뉴 등) 설명",
    "추천 이유를 구체적으로 설명 (분위기·재료·리뷰 등)",
    "위치/접근 정보(강남 어느 쪽·역·교통) 언급",
    "실용 정보(영업시간·예약·주차·웨이팅 등) 최소 1개",
    "예산(4만원)과의 관계 언급(여유·초과 여부)",
]
MAX_SCORE = len(RUBRIC)

def generate(task: str):
    """초안 생성 — 일부러 미니멀(한두 문장). 개선 여지를 남긴다."""
    r = client.chat.completions.create(model=MODEL, messages=[
        {"role": "system", "content": "너는 맛집 추천 도우미다. **한두 문장으로 간단히** 추천만 해라. 부연 설명 금지."},
        {"role": "user", "content": task},
    ])
    return r.choices[0].message.content.strip()

def critique(task: str, answer: str):
    """자기비평 — rubric에서 빠진 항목을 짚고 어떻게 채울지 제안."""
    checklist = "\n".join(f"- {c}" for c in RUBRIC)
    r = client.chat.completions.create(model=MODEL, messages=[
        {"role": "system", "content":
            "너는 깐깐한 편집자다. 아래 체크리스트 중 메시지에서 **빠진 항목**을 모두 짚고, "
            "각 항목을 어떻게 채워넣을지 한 줄씩 구체적으로 제안하라."},
        {"role": "user", "content": f"[체크리스트]\n{checklist}\n\n[과제]\n{task}\n\n[메시지]\n{answer}"},
    ])
    return r.choices[0].message.content.strip()

def improve(task: str, answer: str, feedback: str):
    """비평을 반영해 다시 쓴다."""
    r = client.chat.completions.create(model=MODEL, messages=[
        {"role": "system", "content":
            "너는 맛집 추천 도우미다. 편집자의 피드백을 모두 반영해 메시지를 다시 써라. "
            "체크리스트의 각 항목이 답에 들어가도록 구체적인 정보로 채워라."},
        {"role": "user", "content": f"[과제]\n{task}\n\n[원래 메시지]\n{answer}\n\n[편집자 피드백]\n{feedback}\n\n개선한 메시지:"},
    ])
    return r.choices[0].message.content.strip()

def score(task: str, answer: str):
    """rubric 체크리스트 채점 — 항목별 0/1 판정 후 합. 편차가 작고 개선이 눈에 보임."""
    rubric_text = "\n".join(f"{i+1}. {c}" for i, c in enumerate(RUBRIC))
    r = client.chat.completions.create(model=MODEL, response_format={"type": "json_object"}, messages=[
        {"role": "system", "content":
            "너는 엄격한 평가자다. 아래 rubric 각 항목이 메시지에 **명시적으로 충족**되면 1, 애매하거나 없으면 0. "
            'JSON으로만 답하라: {"items": [{"n": 1, "hit": 0, "why": "…"}, …], "total": 정수}'},
        {"role": "user", "content": f"[Rubric]\n{rubric_text}\n\n[과제]\n{task}\n\n[메시지]\n{answer}"},
    ])
    d = json.loads(r.choices[0].message.content)
    total = int(d.get("total", sum(int(x.get("hit", 0)) for x in d.get("items", []))))
    return max(0, min(MAX_SCORE, total))

# 같은 MVP가 내보낼 추천 메시지를 대상으로
task = "강남에서 2명, 예산 4만원, 매운 음식을 좋아하는 사용자에게 맛집을 추천하는 메시지를 써라. 추천 이유와 1인당 예상 비용을 포함할 것."

v1 = generate(task)
s1 = score(task, v1)
print(f"📝 초안 (점수 {s1}/{MAX_SCORE})\n{v1}\n")

fb = critique(task, v1)
print(f"🔍 자기비평 (rubric 기반 결함 지적)\n{fb}\n")

v2 = improve(task, v1, fb)
s2 = score(task, v2)
print(f"✨ 개선안 (점수 {s2}/{MAX_SCORE})\n{v2}\n")

print(f"📈 품질 변화: {s1} → {s2}/{MAX_SCORE}  ({'향상 ✅' if s2 > s1 else '유지/하락 ⚠️'})")


📝 초안 (점수 2/7)
강남의 '매운 짬뽕'을 추천합니다. 매운 해산물 짬뽕이 유명하며, 1인당 예상 비용은 약 2만원입니다.

🔍 자기비평 (rubric 기반 결함 지적)
메시지에서 빠진 항목들을 체크리스트에 따라 짚고, 각 항목을 어떻게 채워넣을 수 있을지 제안하겠습니다.

1. **구체적인 식당 이름 명시**: 
   - "강남의 '홍콩 짬뽕'이라는 식당을 추천합니다."

2. **매운 정도(맵기 단계·시그니처 메뉴 등) 설명**: 
   - "이 곳의 짬뽕은 5단계로 매운 정도를 조절할 수 있으며, 시그니처 메뉴는 해물 짬뽕입니다."

3. **추천 이유를 구체적으로 설명 (분위기·재료·리뷰 등)**: 
   - "신선한 해산물과 매운 국물의 조화가 일품이며, 고객 리뷰에서도 매운 맛과 풍부한 재료의 조화를 높이 평가하고 있습니다."

4. **위치/접근 정보(강남 어느 쪽·역·교통) 언급**: 
   - "강남역 10번 출구에서 도보로 5분 거리에 위치해 있어 접근성이 매우 좋습니다."

5. **실용 정보(영업시간·예약·주차·웨이팅 등) 최소 1개**: 
   - "영업시간은 매일 오전 11시부터 밤 10시까지이며, 웨이팅은 보통 20분 정도 발생할 수 있으니 참고 바랍니다."

6. **예산(4만원)과의 관계 언급(여유·초과 여부)**: 
   - "1인당 예상 비용이 약 2만원으로, 두 명의 식사 시 총 4만원으로 예산 내에서 충분히 만족할 수 있습니다."

최종 메시지 예시:
"강남의 '홍콩 짬뽕'이라는 식당을 추천합니다. 이 곳의 해물 짬뽕은 5단계로 매운 정도를 조절할 수 있는 시그니처 메뉴이며, 신선한 해산물과 매운 국물의 조화가 일품입니다. 고객 리뷰에서도 높은 평가를 받고 있습니다. 강남역 10번 출구에서 도보로 5분 거리에 위치해 있어 접근성이 매우 좋습니다. 영업시간은 매일 오전 11시부터 밤 10시까지이며, 웨이팅은 보통 20분 정도 발생할 수 있으니 참고 바랍니다. 1인당 예상 비용은 약 2만원으로, 두 명의 식사 시 총 4만

### 정리 + 연습 — 2단계

- **같은 LLM**이 작가이자 편집자입니다(Self-Refine). rubric 점수가 보통 **2~4점** 오릅니다 — 어떤 항목이 새로 채워지는지 눈에 보입니다.
- 반성은 공짜가 아닙니다(호출이 3배 이상). **고위험·고품질이 필요한 곳**에 선별 적용하고 **라운드 수를 제한**하세요.

**🏋️ 연습**: 아래 셀에서 **반성을 2라운드** 돌려 rubric 점수 변화를 비교해 보세요. 어떤 라운드에서 향상이 **수렴**하는지 관찰하세요.


In [10]:
def reflect_loop(task: str, rounds: int = 2):
    """초안 -> (비평 -> 개선) x rounds. rubric 점수 이력을 남긴다."""
    answer = generate(task)
    history = [("초안", answer, score(task, answer))]
    for r in range(1, rounds + 1):
        fb = critique(task, answer)
        answer = improve(task, answer, fb)
        history.append((f"{r}차 개선", answer, score(task, answer)))
    return history

history = reflect_loop(task, rounds=2)

print(f"라운드별 rubric 점수 (만점 {MAX_SCORE})")
prev = None
for label, _ans, sc in history:
    delta = "" if prev is None else f"  ({'+' if sc-prev>=0 else ''}{sc-prev})"
    print(f"  [{label}] {sc}/{MAX_SCORE}{delta}")
    prev = sc

print(f"\n최종 메시지:\n{history[-1][1]}")


라운드별 rubric 점수 (만점 7)
  [초안] 4/7
  [1차 개선] 7/7  (+3)
  [2차 개선] 7/7  (+0)

최종 메시지:
"강남의 '신전 매운탕'을 추천합니다. 이곳은 매운 음식을 좋아하시는 분들께 안성맞춤이며, 1인당 예상 비용은 약 2만원으로, 두 분의 예산 4만원에도 적합합니다. 매운 생선탕은 1인분이 2만원이며, 매운 정도를 1-5단계로 조절할 수 있어 혹시 매운 맛이 부담스러우시다면 낮은 단계로도 주문 가능합니다. 4단계로 드시면 매운맛의 진수를 경험할 수 있어 많은 손님들이 이 메뉴의 깊은 맛과 매운 풍미를 극찬하고 있습니다.

아늑한 분위기와 목재 인테리어로 편안한 식사 환경을 제공하며, 사용되는 모든 재료는 신선하여 건강한 음식을 즐길 수 있습니다. '신전 매운탕'은 강남역 4번 출구에서 도보로 5분 거리에 위치하고 있으며, 인근에 버스 정류장이 있어 대중교통 이용이 편리합니다. 매일 오전 11시부터 밤 10시까지 영업하며, 주말에는 대기 시간이 있을 수 있으니 미리 예약하시면 좋습니다. 전체 예산인 4만원 내에서도 매운 생선탕 2인분과 사이드 메뉴 몇 가지를 여유롭게 즐기실 수 있습니다."


## 3단계 — Self-Consistency: 여러 번 풀어 다수결

강의의 '여러 갈래 사고' — 같은 문제를 temperature를 올려 3번 풀고, 가장 많이 나온 답을 채택합니다.

In [11]:
from collections import Counter

Q = "회의가 오후 2시 40분에 시작해 95분 걸렸다. 끝나는 시각은? '답: HH:MM' 형식 한 줄로 마무리."

def solve_once():
    r = client.chat.completions.create(model=MODEL, temperature=1.0,
        messages=[{"role":"user","content":Q + " 단계별로 생각한 뒤 답하라."}])
    text = r.choices[0].message.content
    return text.strip().splitlines()[-1]     # 마지막 줄(답)만

answers = [solve_once() for _ in range(3)]
print("세 번의 답:", answers)
best, n = Counter(answers).most_common(1)[0]
print(f"다수결 채택: {best} ({n}/3)")
# 관찰: 개별 시도는 흔들려도 다수결은 안정적 — 비용 3배가 대가 (test-time compute)

세 번의 답: ["따라서, 회의가 끝나는 시각은 '답: 16:15'입니다.", '답: 16:15', '답: 16:15']
다수결 채택: 답: 16:15 (2/3)


## 4단계 — CRITIC: 외부 근거로 반성하기

자기비평은 모델이 아는 것 안에서만 돕니다. **외부 근거(문서)**와 대조해 틀린 부분을 고치게 하면 환각을 잡을 수 있습니다.

In [12]:
FACTS = """[우리 서비스 규정]
- 환불 신청: 구매 후 14일 이내
- 배송: 결제 후 2일 내 출고
- 포인트: 구매액의 3% 적립"""

draft = generate("우리 서비스의 환불·배송·포인트 규정을 고객에게 안내하는 문장 3줄")
print("[초안 — 근거 없이 생성]\n", draft)

fixed = client.chat.completions.create(model=MODEL, temperature=0, messages=[
    {"role":"system","content":"아래 근거와 대조해, 초안에서 근거와 다르거나 지어낸 부분을 전부 고쳐라. 고친 뒤 '수정 사항: ...' 한 줄을 덧붙여라."},
    {"role":"user","content":f"<근거>\n{FACTS}\n</근거>\n<초안>\n{draft}\n</초안>"}]).choices[0].message.content
print("\n[CRITIC — 근거 대조 후]\n", fixed)
# 관찰: 자기비평(Self-Refine)이 못 잡는 사실 오류를 외부 근거가 잡는다 — 9·11차시가 반성의 근거로 재등장

[초안 — 근거 없이 생성]
 환불은 구매일로부터 7일 이내에 가능합니다. 배송은 주문 후 3-5일 이내에 진행됩니다. 포인트는 사용 금액의 1%가 적립됩니다.

[CRITIC — 근거 대조 후]
 수정 사항: 환불은 구매일로부터 14일 이내에 가능합니다. 배송은 결제 후 2일 내 출고됩니다. 포인트는 구매액의 3%가 적립됩니다.


## 5단계 — ⭐ 내 팀 MVP에 적용 (통합 설계 결정)

오늘 본 **계획·반성**과 Day2의 부품(도구·RAG·메모리·멀티)을 **day1 MVP의 한 조각**에 끼웁니다. **욕심 금지** — 딱 **한 기능 + 반성 루프**부터.

### ① 우리 MVP의 가장 큰 약점은? → 붙일 기능
| 약점 (증상) | 붙일 기능 | 왜 |
|---|---|---|
| 외부 행동/실시간 정보가 없다 (계산·발송·조회) | **도구(Tool)** | 9차시 — "말하기"에서 "실제로 바꾸기"로 |
| 우리 데이터에 근거하지 못한다 (헛소리) | **RAG** | 11차시 — 내 문서에 grounding |
| 답 품질·정확도가 들쭉날쭉 | **반성(Reflection)** | 오늘 2단계 — 생성→비평→개선 |
| 한 번에 못 푸는 다단계 작업 | **계획(Plan→Execute)** | 오늘 1단계 — 분해해서 끝까지 |

### ② 스코핑 — 좁게 시작하라 
- ✅ 좋은 조각 = **구체적 입력 + 구조화된 출력 + 촘촘한 피드백 루프**
- ❌ "모든 걸 자동화" (엣지케이스 범람) / "만족도 향상" (성공 판단 불가)
- 예: instalite → "게시물 캡션 자동 제안", tastetrail → "여행 메모로 맛집 한 줄 추천"

### ③ 안전 게이트 (Human-in-the-loop)
- **비가역적/고위험 행동(삭제·발송·결제)** 전에는 **사람 승인** 한 단계.
- 실패는 **명확한 사유 + 대안**으로 — 불투명한 거절 금지.


⌨️ **터미널 Codex:** `codex "우리 day1 MVP의 <기능>에 <도구/RAG/반성/계획> 한 조각을 붙이고, 위험 행동 전에 승인 단계를 추가해줘"`

## 정리

- **계획**(나눠서) + **반성**(고쳐서) + **사람 승인**(안전하게) = 끝까지 해내는 에이전트.
- Reflection은 품질을 **측정 가능하게** 올리지만 **비용**이 든다 — 고위험에 선별 적용, 라운드 제한.
- Day2의 부품을 **내 MVP 한 조각**에 끼우면 → **똑똑해진 MVP** (Day2 산출물).